# Feature Engineering & Preprocessing Pipeline

Merges `members`, `transactions`, and `user_logs` into one row per user, engineers the LTV target and behavioral features, then encodes/scales/splits the data so it's ready for the PyTorch `Dataset` in the next notebook.

**Survivorship bias fix**: earlier versions of this pipeline used KKBox's official `train`/`train_v2` label files, which only include users whose subscription happened to be expiring in one fixed snapshot window (Jan/Feb 2017). That silently excludes anyone who churned earlier in the 2015-2016 history and never came back — about 40% of everyone who ever paid KKBox (see `01_EDA.ipynb` discussion). This version instead builds its own population and labels directly from the full `transactions` history: every user's reference date (`ref_date`) is *their own* last observable paid subscription cycle, not a shared global date. A user who churned for good in mid-2015 is now included, using their own 2015 cycle as the reference point, on equal footing with someone still subscribed in late 2016.

Two consequences worth flagging up front:
- **Free trials are excluded from being a reference cycle** — only real (`actual_amount_paid > 0`) transactions establish a `ref_date`, so trial-and-done users don't count as "customer churn."
- **This changes the target rate substantially**: measured across full customer lifetimes instead of a survivors-only snapshot, churn goes from ~9% to ~74%. That's the honest picture the old snapshot was structurally blind to, not a data error.

In [1]:
import json
import os

import duckdb
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = os.path.join(os.getcwd(), "data", "processed")
con = duckdb.connect()


def p(name):
    return os.path.join(PROCESSED_DIR, f"{name}.parquet")


# Per-user reference date replaces the old single global FEATURE_CUTOFF to remove
# survivorship bias: each user's own reference cycle is the latest PAID subscription
# expiration in their own history, not a shared snapshot date. Every feature and label
# below is computed relative to that user's own ref_date, not a global one.
GLOBAL_MAX_DATE = pd.Timestamp("2017-02-28")  # last date covered by transactions/user_logs
LTV_WINDOW_DAYS = 59      # forward revenue window length (matches the original Jan1-Feb28 span)
CHURN_GRACE_DAYS = 30     # KKBox's own definition: no renewal within 30 days of expiry = churn
ENGAGEMENT_WINDOW_DAYS = 30  # trailing listening-activity window ending at each user's ref_date

# Latest a user's reference cycle can end and still leave a full LTV_WINDOW_DAYS of real
# future data to observe (evaluates to 2016-12-31 -- same date the old global cutoff used,
# for the same reason: it's the latest point that still leaves 59 days of runway before
# the data ends on 2017-02-28).
REF_DATE_CUTOFF = GLOBAL_MAX_DATE - pd.Timedelta(days=LTV_WINDOW_DAYS)
print(f"REF_DATE_CUTOFF = {REF_DATE_CUTOFF.date()}")

REF_DATE_CUTOFF = 2016-12-31


## 1. Data merging strategy

Base population: every `msno` with at least one real (`actual_amount_paid > 0`) transaction whose `membership_expire_date` is on/before `REF_DATE_CUTOFF`. That transaction's expiry becomes the user's `ref_date`. All labels and features below are computed relative to each row's own `ref_date`, via a join against a per-user `ref_dates` table rather than a single global date filter.

In [2]:
con.execute(f"""
    create or replace temp table txn as
    select *,
           strptime(cast(transaction_date as varchar), '%Y%m%d')::date as txn_dt,
           strptime(cast(membership_expire_date as varchar), '%Y%m%d')::date as expire_dt
    from '{p("transactions")}'
""")

con.execute(f"""
    create or replace temp table ref_dates as
    -- each user's own last PAID cycle that still leaves a full LTV_WINDOW_DAYS of
    -- observable future data; free-trial ($0) transactions don't establish a ref_date
    select msno, max(expire_dt) as ref_date
    from txn
    where expire_dt <= date '{REF_DATE_CUTOFF.date()}' and actual_amount_paid > 0
    group by msno
""")
n_ref = con.execute("select count(*) from ref_dates").fetchone()[0]
print(f"users with a valid reference cycle: {n_ref:,}")

merge_query = f"""
    with churn_flag as (
        -- KKBox's own definition: churn = no renewal transaction within CHURN_GRACE_DAYS of expiry
        select r.msno,
               case when exists (
                   select 1 from txn t
                   where t.msno = r.msno and t.txn_dt > r.ref_date
                     and t.txn_dt <= r.ref_date + interval {CHURN_GRACE_DAYS} day
               ) then 0 else 1 end as is_churn
        from ref_dates r
    ),
    ltv_target as (
        -- LTV target: forward-looking revenue strictly after each user's own ref_date
        select t.msno, sum(t.actual_amount_paid) as ltv
        from txn t join ref_dates r using (msno)
        where t.txn_dt > r.ref_date and t.txn_dt <= r.ref_date + interval {LTV_WINDOW_DAYS} day
        group by t.msno
    ),
    txn_agg as (
        -- feature aggregates: transactions on/before each user's own ref_date only
        select t.msno,
               count(*) as num_transactions,
               avg(t.payment_plan_days) as avg_payment_plan_days,
               avg(t.actual_amount_paid) as avg_actual_amount_paid,
               avg(t.is_auto_renew) as is_auto_renew_rate
        from txn t join ref_dates r using (msno)
        where t.txn_dt <= r.ref_date
        group by t.msno
    ),
    latest_txn as (
        -- most recent payment method as of each user's own ref_date
        select msno, payment_method_id from (
            select t.msno, t.payment_method_id,
                   row_number() over (partition by t.msno order by t.txn_dt desc) rn
            from txn t join ref_dates r using (msno)
            where t.txn_dt <= r.ref_date
        ) where rn = 1
    ),
    logs_agg as (
        -- total_secs clipped to [0, 86400]: a small fraction of rows have corrupted
        -- extreme values (see 01_EDA.ipynb, "Scanning the full user_logs history" section)
        select l.msno,
               count(distinct l.log_dt) as daily_active_days,
               sum(greatest(least(l.total_secs, 86400), 0)) as total_secs_sum,
               sum(l.num_25) as sum25, sum(l.num_50) as sum50, sum(l.num_75) as sum75,
               sum(l.num_985) as sum985, sum(l.num_100) as sum100
        from (
            select *, strptime(cast(date as varchar), '%Y%m%d')::date as log_dt
            from '{p("user_logs")}'
        ) l
        join ref_dates r using (msno)
        where l.log_dt >= r.ref_date - interval {ENGAGEMENT_WINDOW_DAYS - 1} day
          and l.log_dt <= r.ref_date
        group by l.msno
    )
    select
        r.msno, r.ref_date, cf.is_churn,
        m.city, m.bd, m.gender, m.registered_via, m.registration_init_time,
        coalesce(lt.ltv, 0) as ltv,
        coalesce(txn_agg.num_transactions, 0) as num_transactions,
        txn_agg.avg_payment_plan_days,
        txn_agg.avg_actual_amount_paid,
        coalesce(txn_agg.is_auto_renew_rate, 0) as is_auto_renew_rate,
        latest_txn.payment_method_id,
        coalesce(logs_agg.daily_active_days, 0) as daily_active_days,
        coalesce(logs_agg.total_secs_sum, 0) as total_secs_sum,
        coalesce(logs_agg.sum25, 0) as sum25, coalesce(logs_agg.sum50, 0) as sum50,
        coalesce(logs_agg.sum75, 0) as sum75, coalesce(logs_agg.sum985, 0) as sum985,
        coalesce(logs_agg.sum100, 0) as sum100
    from ref_dates r
    left join churn_flag cf using (msno)
    left join '{p("members")}' m using (msno)
    left join txn_agg using (msno)
    left join latest_txn using (msno)
    left join ltv_target lt using (msno)
    left join logs_agg using (msno)
"""
df = con.execute(merge_query).df()
print(f"merged shape: {df.shape}")
print(f"churn rate: {df['is_churn'].mean():.4f}")
df.isna().sum()

users with a valid reference cycle: 1,610,171


merged shape: (1610171, 21)
churn rate: 0.7360


msno                           0
ref_date                       0
is_churn                       0
city                      192011
bd                        192011
gender                    839131
registered_via            192011
registration_init_time    192011
ltv                            0
num_transactions               0
avg_payment_plan_days       3237
avg_actual_amount_paid      3237
is_auto_renew_rate             0
payment_method_id           3237
daily_active_days              0
total_secs_sum                 0
sum25                          0
sum50                          0
sum75                          0
sum985                         0
sum100                         0
dtype: int64

1. `avg_payment_plan_days` / `avg_actual_amount_paid` / `payment_method_id` are now null only for a small number of rows (~3.2K) where the source `payment_plan_days` field itself is null on the transaction that produced `ref_date` — no longer the ~31.5K "no pre-cutoff transaction" case from the old global-cutoff version, since by construction every row now has at least one qualifying transaction (the one that set its own `ref_date`).
2. `city`/`bd`/`gender`/`registered_via`/`registration_init_time` are null for users absent from `members.parquet` entirely.
3. `gender` is additionally null (missing but present in `members`) for many more, consistent with the ~65% gender-missing rate found in `01_EDA.ipynb`.

## 5.2 Feature engineering

In [3]:
# Registration tenure (days): each user's own ref_date - registration_init_time
reg_date = pd.to_datetime(
    df["registration_init_time"].astype("Int64").astype(str), format="%Y%m%d", errors="coerce"
)
df["registration_tenure_days"] = (pd.to_datetime(df["ref_date"]) - reg_date).dt.days

# Weighted song completion rate
song_totals = df[["sum25", "sum50", "sum75", "sum985", "sum100"]].sum(axis=1)
df["avg_song_completion"] = (
    0.25 * df["sum25"] + 0.50 * df["sum50"] + 0.75 * df["sum75"] + 0.985 * df["sum985"] + 1.0 * df["sum100"]
) / (song_totals + 1)

# Log listening time (30-day pre-ref_date window) and log-LTV target (forward-looking revenue)
df["total_secs_log"] = np.log1p(df["total_secs_sum"])
df["log1p_ltv"] = np.log1p(df["ltv"])

df[["registration_tenure_days", "avg_song_completion", "total_secs_log", "log1p_ltv"]].describe()

,registration_tenure_days,avg_song_completion,total_secs_log,log1p_ltv
count,1.418160e+06,1.610171e+06,1.610171e+06,1.610171e+06
mean,1.202783e+03,6.280897e-01,8.517830e+00,3.160336e+00
std,1.068820e+03,3.605981e-01,4.814518e+00,2.565425e+00
min,-1.717400e+04,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.650000e+02,4.910741e-01,7.153570e+00,0.000000e+00
50%,9.300000e+02,7.908688e-01,1.078058e+01,4.605170e+00
75%,1.740000e+03,8.939672e-01,1.183673e+01,5.293305e+00
max,4.663000e+03,9.999356e-01,1.476593e+01,8.002694e+00


## 5.3 Preprocessing checklist

Split first, then fit every encoder/scaler/imputer statistic **only on the training split** to avoid leakage

In [4]:
df["gender"] = df["gender"].fillna("unknown")
df["bd_clean"] = df["bd"].where(df["bd"].between(1, 100))  # invalid ages imputed after the split

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["is_churn"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["is_churn"], random_state=42)
splits = {"train": train_df, "val": val_df, "test": test_df}

for name, split in splits.items():
    print(f"{name:6s} n={len(split):>7,}  churn_rate={split['is_churn'].mean():.4f}")

train  n=1,127,119  churn_rate=0.7360
val    n=241,526  churn_rate=0.7360
test   n=241,526  churn_rate=0.7360


In [5]:
# Age: median imputation fit on train only
bd_median = train_df["bd_clean"].median()
# Registration tenure: median imputation for the ~11.7% of users absent from members.csv
tenure_median = train_df["registration_tenure_days"].median()
# Transaction aggregates: median imputation for users with no pre-cutoff transaction history
payment_plan_days_median = train_df["avg_payment_plan_days"].median()
actual_amount_paid_median = train_df["avg_actual_amount_paid"].median()
print(f"bd median (train): {bd_median}, tenure median (train): {tenure_median}")
print(f"avg_payment_plan_days median (train): {payment_plan_days_median}, "
      f"avg_actual_amount_paid median (train): {actual_amount_paid_median}")

for split in splits.values():
    split["bd_clean"] = split["bd_clean"].fillna(bd_median)
    split["registration_tenure_days"] = split["registration_tenure_days"].fillna(tenure_median)
    split["avg_payment_plan_days"] = split["avg_payment_plan_days"].fillna(payment_plan_days_median)
    split["avg_actual_amount_paid"] = split["avg_actual_amount_paid"].fillna(actual_amount_paid_median)

bd median (train): 28.0, tenure median (train): 928.0
avg_payment_plan_days median (train): 30.0, avg_actual_amount_paid median (train): 149.0


In [6]:
# Label-encode categoricals to integer indices starting at 0; an extra reserved index
# absorbs missing/unseen categories at val/test/inference time (nn.Embedding-safe).
CATEGORICAL_COLS = ["city", "gender", "registered_via", "payment_method_id"]
# Embedding dims per the plan's Section 3.1 heuristics
EMBEDDING_DIMS = {"city": 5, "gender": 2, "registered_via": 3, "payment_method_id": 8}


def fit_encoder(train_col):
    categories = sorted(train_col.dropna().unique().tolist())
    mapping = {cat: i for i, cat in enumerate(categories)}
    mapping["__unknown__"] = len(categories)
    return mapping


def apply_encoder(col, mapping):
    return col.map(mapping).fillna(mapping["__unknown__"]).astype(int)


encoders = {}
for col in CATEGORICAL_COLS:
    mapping = fit_encoder(train_df[col])
    encoders[col] = mapping
    for split in splits.values():
        split[f"{col}_enc"] = apply_encoder(split[col], mapping)
    print(f"{col:20s} cardinality (incl. unknown bucket): {len(mapping)}")

city                 cardinality (incl. unknown bucket): 22


gender               cardinality (incl. unknown bucket): 4


registered_via       cardinality (incl. unknown bucket): 7


payment_method_id    cardinality (incl. unknown bucket): 40


In [7]:
# StandardScaler on the 7 unbounded numerical columns, fit on train only.
# is_auto_renew_rate and avg_song_completion are already in [0, 1] and left unscaled.
SCALE_COLS = [
    "bd_clean", "registration_tenure_days", "avg_payment_plan_days",
    "avg_actual_amount_paid", "num_transactions", "total_secs_log", "daily_active_days",
]
UNSCALED_NUMERICAL_COLS = ["is_auto_renew_rate", "avg_song_completion"]

scaler = StandardScaler()
scaler.fit(train_df[SCALE_COLS])
for split in splits.values():
    split[[f"{c}_scaled" for c in SCALE_COLS]] = scaler.transform(split[SCALE_COLS])

train_df[[f"{c}_scaled" for c in SCALE_COLS]].describe()

,bd_clean_scaled,registration_tenure_days_scaled,avg_payment_plan_days_scaled,avg_actual_amount_paid_scaled,num_transactions_scaled,total_secs_log_scaled,daily_active_days_scaled
count,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06,1.127119e+06
mean,1.616866e-16,-7.191673e-17,-9.834336e-18,-4.140508e-17,-4.296848e-17,-4.458232e-17,-6.374667e-17
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-4.483081e+00,-1.822778e+01,-6.765974e-01,-7.088030e-01,-1.397655e+00,-1.769357e+00,-1.219612e+00
25%,-1.194013e-01,-7.634828e-01,-2.715103e-01,-3.476114e-01,-9.054808e-01,-2.832588e-01,-1.128138e+00
50%,-1.194013e-01,-2.398023e-01,-2.538978e-01,-2.272142e-01,-4.417656e-02,4.700105e-01,-3.045712e-02
75%,-1.194013e-01,4.229964e-01,-2.538978e-01,-2.272142e-01,6.940842e-01,6.893836e-01,9.757507e-01
max,1.151708e+01,3.471672e+00,5.663896e+00,5.755475e+00,6.477127e+00,1.297437e+00,1.524591e+00


## Save model-ready splits, encoders, and feature manifest

Final feature set: 4 categorical + 9 numerical = 13 columns, matching the plan's Section 5.1 step-6 estimate ("~13 feature columns") exactly.

In [8]:
OUTPUT_COLS = (
    ["msno", "is_churn", "log1p_ltv", "ltv"]
    + [f"{c}_enc" for c in CATEGORICAL_COLS]
    + [f"{c}_scaled" for c in SCALE_COLS]
    + UNSCALED_NUMERICAL_COLS
)

for name, split in splits.items():
    out_path = os.path.join(PROCESSED_DIR, f"model_dataset_{name}.parquet")
    split[OUTPUT_COLS].to_parquet(out_path, engine="pyarrow", compression="zstd", index=False)
    print(f"wrote {out_path} ({len(split):,} rows, {len(OUTPUT_COLS)} cols)")

joblib.dump(encoders, os.path.join(PROCESSED_DIR, "categorical_encoders.joblib"))
joblib.dump(scaler, os.path.join(PROCESSED_DIR, "numerical_scaler.joblib"))

feature_manifest = {
    "methodology": "per-user event-based reference date (fixes survivorship bias vs. a single global snapshot cutoff)",
    "ref_date_max_cutoff": str(REF_DATE_CUTOFF.date()),
    "ltv_window_days": LTV_WINDOW_DAYS,
    "churn_grace_days": CHURN_GRACE_DAYS,
    "engagement_window_days": ENGAGEMENT_WINDOW_DAYS,
    "excludes_free_trial_ref_cycles": True,
    "targets": {"churn": "is_churn", "ltv_log": "log1p_ltv", "ltv_raw": "ltv"},
    "categorical": {
        col: {
            "column": f"{col}_enc",
            "cardinality": len(encoders[col]),
            "embedding_dim": EMBEDDING_DIMS[col],
            "unknown_index": encoders[col]["__unknown__"],
        }
        for col in CATEGORICAL_COLS
    },
    "numerical_scaled": [f"{c}_scaled" for c in SCALE_COLS],
    "numerical_unscaled": UNSCALED_NUMERICAL_COLS,
}
with open(os.path.join(PROCESSED_DIR, "feature_manifest.json"), "w") as f:
    json.dump(feature_manifest, f, indent=2)

feature_manifest

wrote /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/model_dataset_train.parquet (1,127,119 rows, 17 cols)


wrote /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/model_dataset_val.parquet (241,526 rows, 17 cols)


wrote /Users/jaganathapandiyan/Desktop/Python/kkbox-churn-prediction/data/processed/model_dataset_test.parquet (241,526 rows, 17 cols)


{'methodology': 'per-user event-based reference date (fixes survivorship bias vs. a single global snapshot cutoff)',
 'ref_date_max_cutoff': '2016-12-31',
 'ltv_window_days': 59,
 'churn_grace_days': 30,
 'engagement_window_days': 30,
 'excludes_free_trial_ref_cycles': True,
 'targets': {'churn': 'is_churn', 'ltv_log': 'log1p_ltv', 'ltv_raw': 'ltv'},
 'categorical': {'city': {'column': 'city_enc',
   'cardinality': 22,
   'embedding_dim': 5,
   'unknown_index': 21},
  'gender': {'column': 'gender_enc',
   'cardinality': 4,
   'embedding_dim': 2,
   'unknown_index': 3},
  'registered_via': {'column': 'registered_via_enc',
   'cardinality': 7,
   'embedding_dim': 3,
   'unknown_index': 6},
  'payment_method_id': {'column': 'payment_method_id_enc',
   'cardinality': 40,
   'embedding_dim': 8,
   'unknown_index': 39}},
 'numerical_scaled': ['bd_clean_scaled',
  'registration_tenure_days_scaled',
  'avg_payment_plan_days_scaled',
  'avg_actual_amount_paid_scaled',
  'num_transactions_scaled

## Verification

Reload the saved splits from disk and re-check shapes, churn-rate consistency, and null-freeness before treating this pipeline as done.

In [9]:
for name in ["train", "val", "test"]:
    check = pd.read_parquet(os.path.join(PROCESSED_DIR, f"model_dataset_{name}.parquet"))
    n_nulls = check.drop(columns=["msno"]).isna().sum().sum()
    print(
        f"{name:6s} shape={check.shape}  churn_rate={check['is_churn'].mean():.4f}  "
        f"ltv_median={check['ltv'].median():.0f}  total_nulls={n_nulls}"
    )

train  shape=(1127119, 17)  churn_rate=0.7360  ltv_median=99  total_nulls=0
val    shape=(241526, 17)  churn_rate=0.7360  ltv_median=99  total_nulls=0
test   shape=(241526, 17)  churn_rate=0.7360  ltv_median=99  total_nulls=0
